In [ ]:
# !pip install transformers torch ipykernel jupyter ipywidgets

In [ ]:
import torch, pprint
from transformers import pipeline
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)

## Behind the pipeline

In [ ]:
input_seq = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier(input_seq)

In [ ]:
# the default checkpoint used for seq classification
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

inputs = tokenizer(input_seq, padding=True, truncation=True, return_tensors="pt")
print(f"model input or tokenizer output:\n {inputs}")


model = AutoModel.from_pretrained(checkpoint)
outputs = model(**inputs)
print(f"default model output shape: {outputs.last_hidden_state.shape}")


model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)
print(f"model + seq classifier head output shape: {outputs.logits.shape}")
print(f"model + seq classifier head output: {outputs.logits}")


# all HF Transformers models output the logits, as the loss function
# for training will generally fuse the last activation function,
# such as SoftMax, with the actual loss function, such as cross entropy

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(f"model + seq classifier head output after applying softmax: {predictions}")
print(f"seq classifier lebels: {model.config.id2label}")

## Creating a Transformer

### Loading and saving a model checkpoint

In [ ]:
# checkpoint = "bert-base-cased"
# finetuned_ckpt_name = "my-awesome-model"
# user_name = "your-username"
# local_path = "directory_on_my_computer"

# model = AutoModel.from_pretrained(checkpoint)
# # # or, if the arch is known
# # from transformers import BertModel
# # model = BertModel.from_pretrained(checkpoint)

# # ===== save and load locally ==========
# model.save_pretrained(local_path)
# model = AutoModel.from_pretrained(local_path)

# # ===== save and load using HF Hub ====
# from huggingface_hub import notebook_login
# notebook_login() # then, provide the token
# # # or using the CLI
# # !huggingface-cli login
# model.push_to_hub(finetuned_ckpt_name)
# model = AutoModel.from_pretrained(f"{user_name}/{finetuned_ckpt_name}")

### How the tokenizer works?

#### Direct tokenizer `__call__()`

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# # or, if the arch is known
# from transformers import BertTokenizer
# tokenizer = BertTokenizer.from_pretrained(checkpoint)

encoded_input = tokenizer(input_seq)
print("=========================")
pprint.pprint(f"encoded_input (list of lists): {encoded_input}")
pprint.pprint(f"decoded input: {tokenizer.decode(encoded_input['input_ids'])}")

encoded_input = tokenizer(*input_seq, return_tensors="pt")
# without "*" it raises and error.
# with "*" it doesn't behave as expected.
# to solve the problem we have to make them equal-sized. by padding or truncation
print("=========================")
# TODO: run `??tokenizer` and look up return_tensors
pprint.pprint(f"encoded_input (pt tensors): {encoded_input}")

encoded_input = tokenizer(input_seq, return_tensors="pt", padding=True)
print("=========================")
pprint.pprint(f"encoded_input (pt tensors) | with paddings: {encoded_input}")

In [ ]:
encoded_input = tokenizer(
    input_seq,
    padding=True,
    truncation=True,
    max_length=10,
    return_tensors="pt",
)
print("=========================")
pprint.pprint(f"pt tensors | paddings | truncation@10 : {encoded_input}")
pprint.pprint(f"decoded input : {tokenizer.decode(encoded_input['input_ids'])}")

output = model(encoded_input['input_ids'])
print(f"model output using only input ids: {output[0].data}")

output = model(**encoded_input)
print(f"model output using input ids and attn mask: {output[0].data}")

#### `tokenize()` + `encode()` and `decode()`

In [ ]:
print(f"input sequence: {input_seq[0]}")
tokens = tokenizer.tokenize(input_seq[0])
print(f"tokenized sequence/tokens: {tokens}")
ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"encoded tokens to ids: {ids}")
print(f"decoded back to string directly: {tokenizer.decode(ids)}")

### Loading and saving a tokenizer checkpoint

- `tokenizer.save_pretrained` instead of `model.save_pretrained`
- `tokenizer.push_to_hub` instead of `model.push_to_hub`
- `AutoTokenizer.from_pretrained` instead of `AutoModel.from_pretrained`

### Handling multiple sequences

#### Models expect a batch of inputs

In [ ]:
print(f"input sequence: {input_seq[0]}")
tokens = tokenizer.tokenize(input_seq[0])
ids = tokenizer.convert_tokens_to_ids(tokens)
input_ids = torch.tensor(ids)
print(f"input_ids.shape (by tokenize + encode call): {input_ids.shape}")

tokenized_inputs = tokenizer(input_seq[0], return_tensors="pt")
print(f"input_ids.shape (by direct tokenizer call): {tokenized_inputs['input_ids'].shape}")

try: model(input_ids)
except Exception as ex: print("Failed | " + str(ex))

print("after adding a dimension ==============")
input_ids = torch.tensor([ids])
print(f"Input IDs: {input_ids} with shape {input_ids.shape}")
output = model(input_ids)
print("Logits:", output.logits)

#### How attention works?

In [ ]:
sequence1_ids = [[200, 200, 200]]
sequence2_ids = [[200, 200]]
batched_ids = [
    [200, 200, 200],
    [200, 200, tokenizer.pad_token_id],
]

print(model(torch.tensor(sequence1_ids)).logits)
print(model(torch.tensor(sequence2_ids)).logits)
print(model(torch.tensor(batched_ids)).logits)

the second row should be the same as the logits for the second sentence, but we’ve got completely different values!

In [ ]:
attention_mask = [
    [1, 1, 1],
    [1, 1, 0],
]

outputs = model(torch.tensor(batched_ids), attention_mask=torch.tensor(attention_mask))
print(outputs.logits)

Now we get the same logits for the second sentence in the batch.

#### Longer sequences

In [ ]:
model_inputs = tokenizer(input_seq)
print("no pad no truncation===========")
pprint.pprint(model_inputs['attention_mask'])

print("padding to the longest | no truncation")
model_inputs = tokenizer(input_seq, padding="longest")
pprint.pprint(model_inputs['attention_mask'])

max_length=10
print(f"padding up to {max_length} | no truncation")
model_inputs = tokenizer(input_seq, padding="max_length", max_length=max_length)
pprint.pprint(model_inputs['attention_mask'])

print(f"padding up to {max_length} + truncation")
model_inputs = tokenizer(input_seq, padding="max_length", max_length=max_length, truncation=True)
pprint.pprint(model_inputs['attention_mask'])

max_length = tokenizer.model_max_length
print(f"padding up to {max_length} | no truncation")
model_inputs = tokenizer(input_seq, padding="max_length")
print([len(z) for z in model_inputs['attention_mask']])

#### Special Tokens

In [ ]:
model_inputs = tokenizer(input_seq[0])
print(model_inputs["input_ids"])

tokens = tokenizer.tokenize(input_seq[0])
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

print(tokenizer.decode(model_inputs["input_ids"]))
print(tokenizer.decode(ids))